# 2026-09-21：新增 L2 数据接入检查

本轮只核对并隔离重建数据链路，不读取新日期的模型效果，不改变原实验失败结论或验收门槛。研究前提与结论唯一入口为 README 的“新增数据接入检查”。正式存储只读。

## 1. 归档源码与输入

已验证旧实验 1,094 份输入内容没有变化。当前原始成交最晚到 9 月 18 日，但 9 月 17 日下游存在缺口；先归档运行前方案和本轮实际输入。

In [1]:
from pathlib import Path
from datetime import date
import hashlib
import importlib.metadata
import json
import runpy
import shutil
import subprocess
import tarfile
import tempfile
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from src.access import Access, meta
from src.data_system.normalize.level2 import normalize_level2
from src.data_system.builders.level2_stock_trade_1m import build_level2_stock_trade_1m, STOCK_TRADE_1M_KEY
from src.data_system.builders.stock_1430 import (
    build_stock_1430_features, build_stock_1430_labels,
    STOCK_1430_KEY_SCHEMA, STOCK_1430_FEATURE_SCHEMA,
    require_stock_1430_feature_keys,
)
from src.utils.datetime_utils import DateTimeUtils
from src.utils.parquet_writer import write_parquet_atomic
from src.utils.path import PathManager

source = Path('/home/wsw/app/dev/trading')
origin_pm = PathManager(Path('/home/wsw/app/data'))
origin_access = Access(pm=origin_pm, processed_version='v1')
evidence = Path(tempfile.mkdtemp(prefix='subscription-data-update-2026-09-21-', dir='/home/wsw/app/research-evidence'))
print('evidence:', evidence, flush=True)
storage = evidence / 'storage'
storage.mkdir()
pm = PathManager(storage)
access = Access(pm=pm, processed_version='v1')
shutil.copy2(source / 'research/subscription-training/README.md', evidence / 'preregistered-README.md')
shutil.copy2(source / 'research/subscription-training/data_update_validation.ipynb', evidence / 'unexecuted-notebook.ipynb')
with tarfile.open(evidence / 'source.tar', 'w') as archive:
    for item in sorted((source / 'src').rglob('*.py')):
        archive.add(item, arcname=item.relative_to(source))
    for name in ('pyproject.toml', 'uv.lock'):
        archive.add(source / name, arcname=name)

helper = runpy.run_path(str(source / 'research/subscription-training/validate_training.py'))
sz_paths = origin_pm.processed_object(dataset_name='sz_trade', version='v1', trade_date='2026-09-17')
helper['_snapshot_object'](origin_pm, sz_paths, storage)
raw_dir = origin_pm.storage_root / 'raw/level2_ftp/sh_stock_ordertrade/trade_date=2026-09-17'
raw = meta.require(pm=origin_pm, meta_path=raw_dir / 'meta.json')
for item in (raw_dir / 'meta.json', raw.payload_path):
    destination = storage / item.relative_to(origin_pm.storage_root)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(item, destination)
    assert helper['_sha256'](item) == helper['_sha256'](destination)
copied_inputs = [
    {'path': str(item.relative_to(storage)), 'size_bytes': item.stat().st_size, 'sha256': helper['_sha256'](item)}
    for item in sorted(storage.rglob('*')) if item.is_file()
]
(evidence / 'copied-inputs.json').write_text(json.dumps(copied_inputs, indent=2))
run = {
    'base_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=source, text=True).strip(),
    'worktree_status_before': subprocess.check_output(['git', 'status', '--short'], cwd=source, text=True),
    'versions': {name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'pyarrow', 'scikit-learn')},
    'source_storage_read_only': True, 'alpha_evaluated': False,
}
(evidence / 'run.json').write_text(json.dumps(run, ensure_ascii=False, indent=2))
print('copied input files:', len(copied_inputs), flush=True)

evidence: /home/wsw/app/research-evidence/subscription-data-update-2026-09-21-lgrtwnjn


copied input files: 6


## 2. 重建上海 9 月 17 日逐笔

调用现有正常化规则，保留原始过滤、证券分类、市场时段和方向定义。异常必须原样停止，不能删除问题记录或放宽规则。

In [2]:
started = time.monotonic()
raw_copy = storage / raw.payload_path.relative_to(origin_pm.storage_root)
sh_paths = pm.processed_object(dataset_name='sh_trade', version='v1', trade_date='2026-09-17')
normalized = normalize_level2(
    input_file=raw_copy, output_name=sh_paths.payload_path,
    raw_object='SH_Stock_OrderTrade', trade_date='2026-09-17', target_name='sh_trade',
)
write_parquet_atomic(output_file=sh_paths.payload_path, table=normalized.table)
meta.commit(pm=pm, payload_path=sh_paths.payload_path,
            upstream_meta_path=raw_copy.parent / 'meta.json', symbol_slices=normalized.symbol_slices)
normalization_result = {'rows': normalized.table.num_rows,
                        'symbols': len(normalized.symbol_slices),
                        'elapsed_seconds': time.monotonic() - started}
(evidence / 'normalization-result.json').write_text(json.dumps(normalization_result, indent=2))
del normalized
normalization_result

2026-09-21 11:04:33.839 | INFO     | src.data_system.normalize.level2:normalize_level2:165 - ▶️ Level-2 normalize; target=sh_trade trade_date=2026-09-17 input=SH_Stock_OrderTrade.csv.7z


2026-09-21 11:05:03.418 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=30


2026-09-21 11:05:33.428 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=60


2026-09-21 11:06:03.429 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=90


2026-09-21 11:06:33.431 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=120


2026-09-21 11:07:03.441 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=150


2026-09-21 11:07:33.447 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=180


ValueError: trade rows fall outside defined phase intervals: exchange=sh, trade_date=2026-09-17, rows=4187

## 3. 重建两市分钟与 9 月 17 日 Feature

通过隔离存储中的 Access 分批读取，每个 symbol 完整进入既有分钟 builder。随后由 Access 检查两市分钟对象，再生成 Feature。

In [ ]:
minute_results = []
for exchange in ('sh', 'sz'):
    symbols = access.level2_symbols(trade_date='2026-09-17', exchange=exchange)
    parts = []
    for offset in range(0, len(symbols), 100):
        trades = access.trades(trade_date='2026-09-17', symbols=symbols[offset:offset+100], exchange=exchange)
        part = build_level2_stock_trade_1m(pa.concat_tables(list(trades.values())), trade_date='2026-09-17')
        parts.append(part)
    minutes = pa.concat_tables(parts).sort_by([(key, 'ascending') for key in STOCK_TRADE_1M_KEY])
    paths = pm.processed_object(dataset_name=f'{exchange}_stock_trade_1m', version='v1', trade_date='2026-09-17')
    upstream = pm.processed_object(dataset_name=f'{exchange}_trade', version='v1', trade_date='2026-09-17')
    write_parquet_atomic(output_file=paths.payload_path, table=minutes)
    meta.commit(pm=pm, payload_path=paths.payload_path, upstream_meta_path=upstream.meta_path)
    minute_results.append({'exchange': exchange, 'minute_rows': minutes.num_rows})
    del trades, part, parts, minutes
minutes_17 = access.stock_trade_minutes(trade_date='2026-09-17')
feature_17 = build_stock_1430_features(minutes_17, trade_date=date(2026, 9, 17))
feature_path = pm.feature_object(feature_set='l2_stock_1430', version='v1', trade_date='2026-09-17')
write_parquet_atomic(output_file=feature_path.payload_path, table=feature_17)
meta.commit(pm=pm, payload_path=feature_path.payload_path)
pd.DataFrame(minute_results)

## 4. 恢复 9 月 16、17 日监督目标

使用已有 T+1 窗口规则及真实复权因子。只保存标签和覆盖，不读取策略排名表现；9 月 18 日的标签仍需下一个交易日的数据。

In [ ]:
label_results = []
for target in ('2026-09-16', '2026-09-17'):
    following = origin_access.next_trade_date(trade_date=target)
    assert following == ('2026-09-17' if target == '2026-09-16' else '2026-09-18')
    if target == '2026-09-17':
        feature = feature_17
        entry = minutes_17
        exit_minutes = origin_access.stock_trade_minutes(trade_date=following)
        write_parquet_atomic(output_file=evidence / f'input-minutes-{following}.parquet', table=exit_minutes)
    else:
        paths = origin_pm.feature_object(feature_set='l2_stock_1430', version='v1', trade_date=target)
        helper['_snapshot_object'](origin_pm, paths, storage)
        feature = helper['_read_object'](pm, pm.feature_object(feature_set='l2_stock_1430', version='v1', trade_date=target))
        feature = pa.Table.from_pandas(feature, schema=STOCK_1430_FEATURE_SCHEMA, preserve_index=False)
        entry = origin_access.stock_trade_minutes(trade_date=target)
        write_parquet_atomic(output_file=evidence / f'input-minutes-{target}.parquet', table=entry)
        exit_minutes = minutes_17
    factors_t = origin_access.adjustment_factors(trade_date=target)
    factors_t1 = origin_access.adjustment_factors(trade_date=following)
    for day, factors in ((target, factors_t), (following, factors_t1)):
        destination = evidence / f'input-factors-{day}.parquet'
        if not destination.exists():
            write_parquet_atomic(output_file=destination, table=pa.Table.from_pandas(factors, preserve_index=False))
    label = build_stock_1430_labels(
        feature_keys=feature.select(STOCK_1430_KEY_SCHEMA.names),
        entry_minutes=entry, exit_minutes=exit_minutes,
        entry_factors=factors_t, exit_factors=factors_t1,
        trade_date=date.fromisoformat(target), next_trade_date=date.fromisoformat(following),
    )
    assert label.select(STOCK_1430_KEY_SCHEMA.names).equals(feature.select(STOCK_1430_KEY_SCHEMA.names))
    paths = pm.label_object(label_set='l2_stock_1430_t1_vwap_rank', version='v1', trade_date=target)
    write_parquet_atomic(output_file=paths.payload_path, table=label)
    meta.commit(pm=pm, payload_path=paths.payload_path)
    label_results.append({'trade_date': target, 'rows': label.num_rows,
                          'missing_labels': label['y_rank_return'].null_count})
pd.DataFrame(label_results)

## 5. 新日期特征的接入校验

检查日期、完整 key、14:30 决策时点、schema 和特征数值域。独立验证期仍需事前固定候选与通过标准；本轮不读取这些日期的模型效果。

In [ ]:
from datetime import time as wall_time

sessions = origin_access.trade_dates(start_date='2026-08-26', end_date='2026-09-18')
feature_checks = []
for day in sessions:
    if day == '2026-09-17':
        table = feature_17
    else:
        paths = origin_pm.feature_object(feature_set='l2_stock_1430', version='v1', trade_date=day)
        helper['_snapshot_object'](origin_pm, paths, storage)
        record = meta.require(pm=origin_pm, meta_path=paths.meta_path, expected_payload_path=paths.payload_path)
        with pq.ParquetFile(record.payload_path) as parquet:
            table = parquet.read()
    assert table.schema.equals(STOCK_1430_FEATURE_SCHEMA)
    require_stock_1430_feature_keys(table.select(STOCK_1430_KEY_SCHEMA.names), trade_date=day,
        decision_ts_utc=DateTimeUtils.local_time_to_utc_epoch_us(wall_time(14, 30), date.fromisoformat(day)))
    numbers = table.select(STOCK_1430_FEATURE_SCHEMA.names[3:]).to_pandas().to_numpy()
    present = numbers[~np.isnan(numbers)]
    assert np.isfinite(present).all() and ((present >= 0) & (present <= 1)).all()
    feature_checks.append({'trade_date': day, 'rows': table.num_rows, 'schema_key_values_valid': True,
                           'origin': 'isolated reconstruction' if day == '2026-09-17' else 'committed source'})
summary = {
    'new_feature_sessions_checked': len(feature_checks), 'reconstructed_date': '2026-09-17',
    'available_new_target_dates_after_reconstruction': 17,
    'minimum_final_dates_in_H02': 50, 'alpha_evaluated': False,
    'real_reception_readiness_verified': False, 'source_storage_written': False,
    'normalization': normalization_result, 'minute_outputs': minute_results, 'label_outputs': label_results,
}
(evidence / 'feature-checks.json').write_text(json.dumps(feature_checks, indent=2))
(evidence / 'summary.json').write_text(json.dumps(summary, indent=2))
for item in copied_inputs:
    original = origin_pm.storage_root / item['path']
    assert helper['_sha256'](original) == item['sha256']
print('summary:', json.dumps(summary, ensure_ascii=False, indent=2))
pd.DataFrame(feature_checks)